# MIE 402 - Pre-Lab 2: Single Pendulum
**Fall 2026 | 20 points | Data analysis, no coding**

Lab 2 uses high-speed video to measure a single pendulum. You will track the bob, convert its position to angle, and compare motion at several initial angles with a theoretical model.

## What you must submit
Submit **one Word or PDF file** on Canvas before your own laboratory section begins. Include your name, section, GTA, date, calculations, prediction tables, requested figures, and written answers. You may export this completed notebook or assemble the material in Word. Do not submit blank response cells.

## How to use this notebook
Work top to bottom. Complete each prediction before running the analysis cell below it. The code is supplied. You do not need to write, repair, or change Python code. Keep all figures visible. Pre-labs normally appear on Canvas on the Monday before the laboratory; the due time is before your own section begins.


## 1. The physical system and the model you will test
The pivot is fixed. The bob center is a distance $l$ from the pivot. Angle $\theta(t)$ is measured from the downward vertical equilibrium position, and positive angle is chosen toward the right.

The familiar **ideal point-mass model** treats the bar as massless and places all mass $M$ at the bob:
$$Ml^2\ddot{\theta}+Mgl\sin(\theta)=0.$$

The laboratory pendulum has a bar whose mass is not negligible. Model the bar as a uniform slender rod of mass $m$ and length $l$. Its moment of inertia about the pivot is $ml^2/3$, and its weight acts at $l/2$. The **compound-pendulum model** is therefore
$$\left(Ml^2+\frac{ml^2}{3}\right)\ddot{\theta}+\left(Mgl+\frac{mgl}{2}\right)\sin(\theta)=0.$$

For small angles in radians, $\sin(\theta)\approx\theta$:
$$\omega_n^2=\frac{g(M+m/2)}{l(M+m/3)},\qquad f_n=\frac{\omega_n}{2\pi},\qquad T_0=\frac{2\pi}{\omega_n}.$$

**Task 1.** Use $l=0.82$ m, bob mass $M=0.300$ kg, bar mass $m=0.120$ kg, and $g=9.81$ m/s$^2$. Calculate $\omega_n$, $f_n$, and $T_0$ for both the ideal point-mass model and the compound-pendulum model. Report the percent difference in $f_n$. Explain why mass cancels in the ideal model but the mass distribution matters in the real apparatus.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
g=9.81; ell=0.82; M=0.300; m=0.120
omega_point=np.sqrt(g/ell)
omega_n=np.sqrt(g*(M+m/2)/(ell*(M+m/3)))
f_point=omega_point/(2*np.pi); f_n=omega_n/(2*np.pi); T0=2*np.pi/omega_n
print(f'Point-mass model: omega_n={omega_point:.3f} rad/s, f_n={f_point:.3f} Hz, T={2*np.pi/omega_point:.3f} s')
print(f'Compound model:   omega_n={omega_n:.3f} rad/s, f_n={f_n:.3f} Hz, T={T0:.3f} s')
print(f'Frequency difference = {100*(f_n-f_point)/f_point:.2f}%')


## 2. Predict before running the simulation
The planned initial angles are 8, 35, and 105 degrees. Every trial starts from rest.

**Task 2.** Before running the next cell, complete this table in your submitted file.

| Initial angle | Is small-angle theory appropriate? | Period compared with $T_0$ | Maximum speed compared with 8 degrees |
|---|---|---|---|
| 8 degrees | | | |
| 35 degrees | | | |
| 105 degrees | | | |


In [ ]:
# Supplied nonlinear simulation. Do not modify.
def simulate(theta0_deg,dt=0.001,duration=16):
 n=int(duration/dt)+1;t=np.arange(n)*dt;theta=np.zeros(n);omega=np.zeros(n);theta[0]=np.deg2rad(theta0_deg)
 alpha=g*(M+m/2)/(ell*(M+m/3))
 def rhs(q):return np.array([q[1],-alpha*np.sin(q[0])])
 q=np.array([theta[0],0.0])
 for i in range(n-1):
  k1=rhs(q);k2=rhs(q+dt*k1/2);k3=rhs(q+dt*k2/2);k4=rhs(q+dt*k3)
  q=q+dt*(k1+2*k2+2*k3+k4)/6;theta[i+1],omega[i+1]=q
 return t,theta,omega
fig,ax=plt.subplots(3,1,figsize=(9,8),sharex=True)
for angle,axis in zip([8,35,105],ax):
 t,th,w=simulate(angle);axis.plot(t,np.rad2deg(th));axis.grid(True);axis.set_ylabel('theta (deg)');axis.set_title(f'theta(0) = {angle} deg')
ax[-1].set_xlabel('time (s)');plt.tight_layout()


**Task 3.** Estimate one period for each case from the plots. Make a table with your estimates and $T_0$. Which case agrees most closely with small-angle theory? Why does the 105-degree case take longer even though $L$ has not changed?

## 3. From camera coordinates to angle
The camera does not measure angle directly. The tracking app returns pivot location $(x_p,y_p)$ and bob location $(x_b,y_b)$ for each frame. Use:
For standard image coordinates, $x$ increases to the right and $y$ increases downward. Then
$$\theta=\operatorname{atan2}(x_b-x_p,\;y_b-y_p).$$
This definition gives $\theta=0$ when the bob hangs directly below the pivot. The atan2 function preserves the quadrant. If your exported coordinate convention differs, verify the sign and zero angle using a frame whose physical orientation you know.

**Task 4.** Explain why a fixed pivot location and good pixel calibration matter. What happens if the bob leaves the video frame?


In [ ]:
t,theta,omega=simulate(35,duration=8)
xp,yp,Lmm=500.,120.,820.
xb=xp+Lmm*np.sin(theta);yb=yp+Lmm*np.cos(theta)
theta_xy=np.arctan2(xb-xp,yb-yp)
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(xb,yb);ax[0].plot(xp,yp,'ro',label='pivot');ax[0].set_aspect('equal');ax[0].invert_yaxis();ax[0].set_xlabel('x (mm)');ax[0].set_ylabel('y (mm)');ax[0].legend();ax[0].set_title('Tracked bob path')
ax[1].plot(t,np.rad2deg(theta_xy));ax[1].grid(True);ax[1].set_xlabel('time (s)');ax[1].set_ylabel('theta (deg)');ax[1].set_title('Angle recovered from x and y');plt.tight_layout()


## 4. Frame rate and frequency
The camera normally records at 1000 frames/s. The pendulum frequency is near 0.55 Hz, so each cycle contains many frames. This rate greatly exceeds the Nyquist minimum for the pendulum motion, but it resolves the release and supplies closely spaced samples for tracking and numerical differentiation.

**Task 5.** Calculate the Nyquist frequency for 1000 frames/s. Approximately how many frames occur in one small-angle period? Explain why actual frame rate belongs in lab notes.


In [ ]:
camera_fs=1000
print(f'Camera Nyquist frequency = {camera_fs/2:.0f} Hz')
print(f'Frames in one small-angle period = {camera_fs*T0:.0f}')
t,theta,omega=simulate(35,duration=16);dt=t[1]-t[0];N=len(theta)
P=np.abs(np.fft.rfft(theta-theta.mean()))/N;P[1:-1]*=2;f=np.fft.rfftfreq(N,dt)
plt.figure(figsize=(8,4));plt.plot(f,P);plt.xlim(0,3);plt.grid(True);plt.xlabel('frequency (Hz)');plt.ylabel('one-sided amplitude (rad)');plt.title('Simulated 35-degree pendulum spectrum')
print(f'FFT-bin spacing = {1/(N*dt):.4f} Hz')


## 5. Plan the laboratory record
**Task 6.** Write a short plan covering: how the group measures $l$, $M$, and $m$; how it sets and records each initial angle; camera settings checked before recording; quantities recorded from the tracking result; and at least two data-quality checks before leaving.

Mention all three Lab 2 ranges: below 10 degrees, 10 to 90 degrees, and 90 to 180 degrees. State that every release starts from rest.
